# Airbnb Paris – Semantische Embeddings
- Jede Freitextzelle mit `all-mpnet-base-v2` eingebettet, Spaltennamen-Embedding addiert, danach LayerNorm je Vektor
- Seed-unabhängig: das Modell ist vortrainiert und eingefroren, es wird nichts gefittet
- Ausgabe als float32-Parquet (`row_id` + Embeddings), die PCA folgt in `semantic_pca.ipynb`

In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

TEXT_COLS = ["name", "description", "neighborhood_overview", "host_about"]

## Texte laden & Modell initialisieren

In [ ]:
df = pd.read_csv("../../data/preprocessed/cleaned_text_airbnb_paris.csv", keep_default_na=False)
model = SentenceTransformer("all-mpnet-base-v2")
name_emb = {c: model.encode(c) for c in TEXT_COLS}  # cached, constant per column

## Zellen einbetten, Spaltennamen addieren, LayerNorm
- Die Normalisierung läuft **je Zeile** über die Embedding-Dimensionen, nicht über den Datensatz

In [ ]:
parts = []
for c in TEXT_COLS:
    e = model.encode(df[c].astype(str).tolist(), batch_size=64, show_progress_bar=False) + name_emb[c]
    e = (e - e.mean(axis=1, keepdims=True)) / (e.std(axis=1, keepdims=True) + 1e-6)
    cols = [f"{c}_emb_{i}" for i in range(e.shape[1])]
    parts.append(pd.DataFrame(e.astype("float32"), columns=cols, index=df.index))

## Speichern

In [ ]:
out = pd.concat([df[["row_id"]]] + parts, axis=1)
out.to_parquet("../../data/preprocessed/semantic_emb_airbnb_paris.parquet", index=False)
print("Shape:", out.shape, "| Embedding-Spalten:", out.shape[1] - 1)

## Verifikation

In [ ]:
assert out["row_id"].is_unique
assert out.isna().sum().sum() == 0